# Everything

Every node type that emits, wired up once.

*Exported from Coda v0.0.0-test on 2026-01-01.*

In [ ]:
# pip install flybrains matplotlib navis navis-fastcore networkx neuprint-python numpy pandas scipy seaborn

import os
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import navis
import navis.interfaces.neuprint as neu
import navis_fastcore as fastcore
import flybrains  # noqa: F401  (registers the fly templates)
from neuprint import Client, NeuronCriteria, SynapseCriteria, connection_table_to_matrix, fetch_adjacencies, fetch_custom, fetch_neurons, fetch_paths, fetch_primary_rois, fetch_synapses, merge_neuron_properties
from scipy.cluster.hierarchy import cut_tree, dendrogram, fcluster, leaves_list, linkage
from scipy.spatial.distance import squareform

In [ ]:
# Helpers, generated by Coda. These are the parts of the workflow that have
# no equivalent in the libraries this notebook imports, written out here so
# it stands on its own.

def coda_combine(df, columns, source=False):
    """The first of `columns` holding a value per row, or which column that was."""
    out = pd.Series([None] * len(df), index=df.index, dtype=object)
    for name in columns:
        if name not in df.columns:
            continue
        col = df[name]
        # Null and the empty string are one absence: a blank must not stop the search.
        have = col.notna() & (col.astype(str) != '')
        out = out.mask(out.isna() & have, name if source else col)
    return out


def coda_int64(values):
    """A column of ids as exact int64. Anything unreadable becomes 0."""

    def one(value):
        try:
            # Exact for an int and for decimal text of any width, which pd.to_numeric is
            # not: a single null in the column makes it answer float64, and an
            # eighteen-digit root id through a float is a different neuron.
            return int(value)
        except (TypeError, ValueError):
            try:
                return int(float(value))
            except (TypeError, ValueError):
                return 0

    return values.map(one).astype('int64')


def coda_annotation_columns(df, id_column):
    """Rename a CAVE table's columns to the two names Coda addresses by name."""
    renames = {}
    if id_column in df.columns:
        renames[id_column] = 'neuronId'
    for name in ('cell_type', 'celltype'):
        if name in df.columns and 'type' not in df.columns:
            renames[name] = 'type'
            break
    out = df.rename(columns=renames)
    if 'neuronId' not in out.columns:
        return out
    # A row with no id names no neuron, which is what Coda's shaping drops. An empty
    # string counts: SeaTable spells a blank cell that way.
    ids = out['neuronId']
    out = out[ids.notna() & (ids.astype(str) != '')].copy()
    ids = out['neuronId']
    if pd.api.types.is_float_dtype(ids):
        # Already lossy — a float64 cannot hold an eighteen-digit root id — but `str()` of
        # one is `7.2e+17`, which matches nothing at all. Integer text at least keeps the
        # shape of an id. Text and int columns go straight through, exact at any width.
        ids = coda_int64(ids)
    out['neuronId'] = ids.astype(str)
    return out


def coda_google_sheet(url, id_column='root_id', columns=None):
    """A shared Google Sheet, read through its CSV export URL."""
    # `dtype=` rather than a cast afterwards: a float64 id is already the wrong neuron by
    # the time you could cast it back.
    df = pd.read_csv(url, dtype={id_column: 'string'})
    if id_column not in df.columns:
        raise KeyError(
            f"{id_column!r} is not a column of that tab. It has: {list(df.columns)}"
        )
    if columns:
        keep = [c for c in columns if c in df.columns and c != id_column]
    else:
        # Empty means every column but the id, which is how a sheet says "all of it".
        keep = [c for c in df.columns if c != id_column]
    return coda_annotation_columns(df[[id_column] + keep], id_column)


def coda_join(values):
    """Coda's `join` aggregation: distinct, first-appearance order, absences skipped."""
    kept = dict.fromkeys(str(v) for v in values.dropna() if str(v) != '')
    return "; ".join(kept) if kept else None


def coda_join_annotations(left, right):
    """Chain two annotation sources: outer join on `neuronId`, the later one winning."""
    if left is None:
        return right
    if right is None:
        return left
    left = left.drop_duplicates(subset=['neuronId'], keep='first')
    right = right.drop_duplicates(subset=['neuronId'], keep='first')
    shared = [c for c in right.columns if c in left.columns and c != 'neuronId']
    merged = left.merge(
        right, on='neuronId', how='outer', suffixes=('', '_coda_later')
    )
    for name in shared:
        later = merged[name + '_coda_later']
        # Later wins, falling back to the earlier source where the later one is null.
        merged[name] = later.combine_first(merged[name])
        merged = merged.drop(columns=[name + '_coda_later'])
    return merged


def coda_neurons(df):
    """Rename neuprint-python's `bodyId` to the `neuronId` every Coda table uses."""
    if df is None or 'bodyId' not in df.columns or 'neuronId' in df.columns:
        return df
    return df.rename(columns={'bodyId': 'neuronId'})


import re

_CODA_ROI_SIDE = re.compile(r"\((L|R)\)\s*$")


def _coda_roi_side(roi):
    """Which side of the animal an ROI sits on.

    One rule rather than a per-dataset table: every neuPrint dataset writes the side as a
    trailing parenthesis, and a name without one -- "ANm", "CV" -- is genuinely
    unlateralised rather than unlabelled. Reads the LAST parenthesis deliberately:
    "HTct(UTct-T3)(L)" has two, and anchoring on the first reports every leg neuropil
    as unsided.
    """
    match = _CODA_ROI_SIDE.search(roi or "")
    return match.group(1) if match else "center"


def _coda_partner_types(conn, top_n):
    """Partners rolled up by type, with both shares.

    Synapses summed *and* distinct partners counted in one pass, because the two answer
    different questions -- forty synapses onto one neuron is not forty onto forty.
    Untyped partners keep their own bucket rather than folding into a neighbour: on
    male-CNS a large share of a neuron's partners are untyped, and merging them silently
    puts a fictitious type at the top of the list.
    """
    if conn.empty:
        return conn.assign(type=None, synapses=0, partners=0,
                           synapse_share=0.0, partner_share=0.0).iloc[0:0]

    out = []
    for body, group in conn.groupby("neuronId", sort=False):
        total_syn = group["weight"].sum()
        total_partners = group["partnerId"].nunique()
        rolled = (
            group.groupby("partnerType", dropna=False)
            .agg(synapses=("weight", "sum"), partners=("partnerId", "nunique"))
            .reset_index()
            .rename(columns={"partnerType": "type"})
        )
        rolled["neuronId"] = body
        rolled["synapse_share"] = rolled["synapses"] / total_syn if total_syn else 0.0
        rolled["partner_share"] = (
            rolled["partners"] / total_partners if total_partners else 0.0
        )
        rolled = rolled.sort_values(
            ["synapses", "type"], ascending=[False, True], na_position="last",
        )
        out.append(rolled.head(top_n) if top_n else rolled)

    cols = ["neuronId", "type", "synapses", "partners", "synapse_share", "partner_share"]
    return pd.concat(out, ignore_index=True)[cols]


def _coda_top_partners(conn, top_n):
    """Individual partner neurons, strongest first, with each one's share."""
    if conn.empty:
        return conn.assign(share=0.0).iloc[0:0]

    out = []
    for body, group in conn.groupby("neuronId", sort=False):
        total = group["weight"].sum()
        rows = group.copy()
        rows["share"] = rows["weight"] / total if total else 0.0
        rows = rows.sort_values(["weight", "partnerId"], ascending=[False, True])
        out.append(rows.head(top_n) if top_n else rows)

    cols = ["neuronId", "partnerId", "partnerType", "weight", "share"]
    return pd.concat(out, ignore_index=True)[cols]


def _coda_connectivity(neuron_ids, direction, min_weight, client):
    """One direction of partners, in the query-relative shape the roll-ups expect.

    neuronId is always the neuron being profiled and partnerId is whatever it is wired to,
    whichever way the arrow points -- which is the *opposite* convention to Coda's
    Connectivity node, and the right one here: "these are my upstream partners" is the
    question a profile asks.
    """
    criteria = NeuronCriteria(bodyId=list(neuron_ids), client=client)
    if direction == "downstream":
        neurons, conn = fetch_adjacencies(
            criteria, None, min_total_weight=min_weight,
            omit_rois=True, client=client,
        )
        mine, theirs = "bodyId_pre", "bodyId_post"
    else:
        neurons, conn = fetch_adjacencies(
            None, criteria, min_total_weight=min_weight,
            omit_rois=True, client=client,
        )
        mine, theirs = "bodyId_post", "bodyId_pre"

    if conn.empty:
        return pd.DataFrame(
            columns=["neuronId", "partnerId", "partnerType", "weight"]
        )

    conn = merge_neuron_properties(neurons, conn, ["type"])
    partner_type = "type_post" if mine == "bodyId_pre" else "type_pre"
    return conn.rename(columns={
        mine: "neuronId", theirs: "partnerId", partner_type: "partnerType",
    })[["neuronId", "partnerId", "partnerType", "weight"]]


def coda_profile(neuron_ids, client, min_weight=1, top_n=10):
    """Everything Coda's Neuron Profile card shows, as a dict of DataFrames.

    Keys mirror the tiles: summary, upstream_types, downstream_types, top_upstream,
    top_downstream, regions, hemispheres.

    Three requests regardless of how many neurons are asked for, because every fetch here
    takes the whole id list -- so this is cheap to widen from the pinned neuron to the
    entire table, which is the one thing the Neuron Profile widget cannot do.

    min_weight drops connections below a threshold; top_n caps each list (0 keeps all).
    """
    neuron_ids = [int(b) for b in neuron_ids]
    if not neuron_ids:
        empty = pd.DataFrame()
        return {k: empty for k in ("summary", "upstream_types", "downstream_types",
                                   "top_upstream", "top_downstream", "regions",
                                   "hemispheres")}

    neurons, roi_counts = fetch_neurons(
        NeuronCriteria(bodyId=neuron_ids, client=client), client=client,
    )
    neurons, roi_counts = coda_neurons(neurons), coda_neurons(roi_counts)
    up = _coda_connectivity(neuron_ids, "upstream", min_weight, client)
    down = _coda_connectivity(neuron_ids, "downstream", min_weight, client)

    # roiInfo NESTS: a synapse in LO(R) is counted again in its parent OL(R), so summing
    # the raw breakdown reports roughly twice the neuron's synapses. Only the primary ROIs
    # tile the volume, which is why they are fetched rather than assumed.
    primary = set(fetch_primary_rois(client=client))
    regions = roi_counts[roi_counts["roi"].isin(primary)].copy()
    regions = (
        regions.groupby(["neuronId", "roi"], as_index=False)[["pre", "post"]].sum()
    )
    regions["total"] = regions["pre"] + regions["post"]
    regions = regions[regions["total"] > 0].sort_values(
        ["neuronId", "total", "roi"], ascending=[True, False, True],
    )

    sides = regions.assign(side=regions["roi"].map(_coda_roi_side))
    hemispheres = (
        sides.pivot_table(index="neuronId", columns="side", values="total",
                          aggfunc="sum", fill_value=0)
        .rename(columns={"L": "left", "R": "right"})
        .reset_index()
    )
    for column in ("left", "right", "center"):
        if column not in hemispheres:
            hemispheres[column] = 0
    hemispheres["total"] = (
        hemispheres["left"] + hemispheres["right"] + hemispheres["center"]
    )

    def totals(conn, prefix):
        if conn.empty:
            return pd.DataFrame(columns=["neuronId", f"{prefix}_synapses",
                                         f"{prefix}_partners"])
        return (
            conn.groupby("neuronId", as_index=False)
            .agg(**{f"{prefix}_synapses": ("weight", "sum"),
                    f"{prefix}_partners": ("partnerId", "nunique")})
        )

    summary = neurons.merge(totals(up, "upstream"), on="neuronId", how="left")
    summary = summary.merge(totals(down, "downstream"), on="neuronId", how="left")
    summary = summary.merge(
        hemispheres[["neuronId", "left", "right", "center"]], on="neuronId", how="left",
    )

    return {
        "summary": summary,
        "upstream_types": _coda_partner_types(up, top_n),
        "downstream_types": _coda_partner_types(down, top_n),
        "top_upstream": _coda_top_partners(up, top_n),
        "top_downstream": _coda_top_partners(down, top_n),
        "regions": regions,
        "hemispheres": hemispheres,
    }


def _coda_rng(seed):
    """mulberry32, the generator behind Coda's Sample node."""
    a = int(seed) & 0xFFFFFFFF

    def rand():
        nonlocal a
        a = (a + 0x6D2B79F5) & 0xFFFFFFFF
        t = a
        t = ((t ^ (t >> 15)) * (t | 1)) & 0xFFFFFFFF
        t = (t ^ (t + (((t ^ (t >> 7)) * (t | 61)) & 0xFFFFFFFF))) & 0xFFFFFFFF
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296

    return rand


def coda_sample_rows(length, count, seed):
    """Row positions for a seeded draw, ascending.

    Partial Fisher-Yates over `count` draws, then sorted: this samples rather than
    shuffles, so a random subset of a sorted table stays sorted.
    """
    length = max(0, int(length))
    count = max(0, min(length, int(count)))
    idx = list(range(length))
    rand = _coda_rng(seed)
    for i in range(count):
        j = i + int(rand() * (length - i))
        idx[i], idx[j] = idx[j], idx[i]
    return sorted(idx[:count])


import re

_CODA_OPERATORS = [("==", "eq"), ("!=", "ne"), (">=", "ge"), ("<=", "le"),
                   ("~", "match"), (">", "gt"), ("<", "lt"), ("=", "eq")]
_CODA_FIELD_NAME = re.compile(r"^[A-Za-z_][A-Za-z0-9_.]*$")


def _coda_tokenize(text):
    """Whitespace-split, but quotes hold a token together."""
    tokens, i = [], 0
    while i < len(text):
        while i < len(text) and text[i].isspace():
            i += 1
        if i >= len(text):
            break
        start, quote = i, None
        while i < len(text):
            ch = text[i]
            if quote is not None:
                if ch == quote:
                    quote = None
            elif ch in "\"'":
                quote = ch
            elif ch.isspace():
                break
            i += 1
        tokens.append(text[start:i])
    return tokens


def _coda_unquote(value):
    if value[:1] in ("\"", "'") and len(value) >= 2:
        return value[1:-1] if value.endswith(value[0]) else value[1:]
    return value


def _coda_split_operator(token):
    """Field/operator/value, or None for a bare word.

    Operators are tried longest-first so "!=" is not read as "=", and the field has to
    look like a name -- otherwise "LC4-a" would parse as a comparison.
    """
    for symbol, op in _CODA_OPERATORS:
        at = token.find(symbol)
        if at <= 0:
            continue
        field = token[:at]
        if not _CODA_FIELD_NAME.match(field):
            continue
        return field, op, token[at + len(symbol):]
    return None


def _coda_parse_search(text):
    terms = []
    for raw in _coda_tokenize(text):
        negate = False
        if raw[:1] in ("!", "-") and len(raw) > 1 and _coda_split_operator(raw) is None:
            negate, raw = True, raw[1:]
        split = _coda_split_operator(raw)
        if split is None:
            value = _coda_unquote(raw)
            if value:
                terms.append(("text", value.lower(), None, None, negate))
            continue
        field, op, value = split
        value = _coda_unquote(value)
        if not value:
            # Every query mid-typing looks like this; it narrows nothing rather than
            # being an error.
            continue
        terms.append(("field", value, field, op, negate))
    return terms


def _coda_haystack(df):
    """Lowercase text of every searchable column, one string per row.

    String columns and neuronId only -- so a bare "1200" finds a neuron id and does not
    also match every neuron with 1200 synapses.
    """
    cols = [c for c in df.columns
            if df[c].dtype == object or str(c) == "neuronId"]
    if not cols:
        return pd.Series([""] * len(df), index=df.index)
    parts = [df[c].fillna("").astype(str) for c in cols]
    joined = parts[0]
    for part in parts[1:]:
        joined = joined.str.cat(part, sep=" ")
    return joined.str.lower()


def _coda_field_mask(df, field, op, value):
    """One field comparison.

    A missing value satisfies "!=" and nothing else -- so status!=Traced returns the
    untraced *and* the unlabelled, which is the question somebody auditing a dataset
    for gaps is actually asking. SQL's three-valued logic drops both, silently.
    """
    col = next((c for c in df.columns if str(c).lower() == field.lower()), None)
    if col is None:
        return pd.Series(False, index=df.index)
    series = df[col]
    missing = series.isna()

    if op == "match":
        # Unanchored, deliberately unlike neuPrint's "=~": this search is local and has
        # no server semantic to match.
        rx = re.compile(value)
        found = series.fillna("").astype(str).map(lambda v: rx.search(v) is not None)
        return found & ~missing

    if pd.api.types.is_numeric_dtype(series):
        try:
            right = float(value)
        except ValueError:
            return pd.Series(False, index=df.index)
        left = pd.to_numeric(series, errors="coerce")
    else:
        right = value.lower()
        left = series.fillna("").astype(str).str.lower()

    if op == "eq":
        mask = left == right
    elif op == "ne":
        mask = left != right
    elif op == "gt":
        mask = left > right
    elif op == "lt":
        mask = left < right
    elif op == "ge":
        mask = left >= right
    else:
        mask = left <= right

    mask = mask.fillna(False).astype(bool)
    return (mask | missing) if op == "ne" else (mask & ~missing)


def coda_search(df, query):
    """Rows matching Coda's Explore Dataset query language.

    Terms are AND-ed; a leading "!" or "-" negates one. A bare word is a substring of
    the row's searchable text; "field=value" compares one column, with ">" "<" ">=",
    "<=", "!=" and "~" (unanchored regex) as the other operators.

    Two things this does NOT reproduce, both of which change which rows you get:

    * Hits come back in table order. Coda ranks them by relevance, which only matters
      where the result is capped -- but there it decides which rows survive the cap.
    * A query matching nothing returns nothing. Coda retries it as a subsequence, so
      "mechnosensory" still finds "mechanosensory" there and finds nothing here.
    """
    terms = _coda_parse_search(query)
    if not terms:
        return df

    keep = pd.Series(True, index=df.index)
    haystack = None
    for kind, value, field, op, negate in terms:
        if kind == "text":
            if haystack is None:
                haystack = _coda_haystack(df)
            mask = haystack.str.contains(value, regex=False)
        else:
            mask = _coda_field_mask(df, field, op, value)
        keep &= ~mask if negate else mask

    return df[keep]


def coda_synapses(df):
    """neuprint-python synapse columns as Coda names them: `type` is the polarity."""
    if df is None or 'type' not in df.columns or 'polarity' in df.columns:
        return df
    return df.rename(columns={'type': 'polarity'})


def coda_traverse_connectivity(seed_ids, direction, hops, min_weight, client):
    """Coda's Connectivity node: a breadth-first walk returning an edge list.

    Columns are preId/preType -> postId/postType, weight, hop, direction. Every row is
    oriented the way the synapse points, whichever way the traversal travelled.

    Three rules worth keeping, each of which silently changes the answer if dropped:

    * A neuron is expanded at most once. Connectomes are full of recurrent loops, so a
      walk that re-expands a visited neuron does not terminate. The edge back into an
      already-visited neuron is still reported; only the expansion is skipped.
    * An edge re-found at a later hop keeps the hop and direction it was first given,
      so the label says something about the graph rather than about the walk order.
    * direction="both" expands both ways at every hop -- the undirected ball, not two
      cones. That is what finds the neurons sharing input with a seed.
    """
    def one_hop(ids, way):
        criteria = NeuronCriteria(bodyId=list(ids), client=client)
        if way == "downstream":
            neurons, conn = fetch_adjacencies(
                criteria, None, min_total_weight=min_weight,
                omit_rois=True, client=client,
            )
        else:
            neurons, conn = fetch_adjacencies(
                None, criteria, min_total_weight=min_weight,
                omit_rois=True, client=client,
            )
        if conn.empty:
            return conn
        return merge_neuron_properties(neurons, conn, ["type"]).assign(direction=way)

    ways = ["downstream", "upstream"] if direction == "both" else (
        ["upstream"] if direction == "inputs" else ["downstream"]
    )

    frontier = set(int(i) for i in seed_ids)
    expanded = set()
    rows = []
    seen_edges = {}

    for hop in range(1, int(hops) + 1):
        if not frontier:
            break
        todo = frontier - expanded
        if not todo:
            break
        expanded |= todo

        found = [one_hop(todo, way) for way in ways]
        found = [f for f in found if not f.empty]
        if not found:
            break
        step = pd.concat(found, ignore_index=True)

        next_frontier = set()
        for row in step.itertuples(index=False):
            key = (int(row.bodyId_pre), int(row.bodyId_post))
            if key in seen_edges:
                # Reached from the other end at this same hop: an edge internal to the
                # frontier, which is the one case that earns the "both" label.
                if seen_edges[key][0] == hop and seen_edges[key][1] != row.direction:
                    seen_edges[key] = (hop, "both")
                continue
            seen_edges[key] = (hop, row.direction)
            rows.append(row._asdict())
            next_frontier.add(key[1] if row.direction == "downstream" else key[0])

        frontier = next_frontier - expanded

    if not rows:
        return pd.DataFrame(
            columns=["preId", "preType", "postId", "postType",
                     "weight", "hop", "direction"]
        )

    out = pd.DataFrame(rows)
    resolved = [seen_edges[(int(p), int(q))]
                for p, q in zip(out["bodyId_pre"], out["bodyId_post"])]
    out["hop"] = [r[0] for r in resolved]
    out["direction"] = [r[1] for r in resolved]
    return out.rename(columns={
        "bodyId_pre": "preId",
        "type_pre": "preType",
        "bodyId_post": "postId",
        "type_post": "postType",
    })

In [ ]:
# ── Hemibrain (neuPrint) ──
hemibrain_neuprint = Client(
    'neuprint.janelia.org',
    dataset='hemibrain:v1.2.1',
    token=os.environ['NEUPRINT_APPLICATION_CREDENTIALS'],
)

In [ ]:
# ── Custom neuPrint ──
custom_neuprint = Client(
    'neuprint.janelia.org',
    dataset='manc:v1.2.3',
    token=os.environ['NEUPRINT_APPLICATION_CREDENTIALS'],
)

In [ ]:
# ── MaleCNS (neuPrint) ──
malecns_neuprint = Client(
    'neuprint.janelia.org',
    dataset='male-cns:v1.0',
    token=os.environ['NEUPRINT_APPLICATION_CREDENTIALS'],
)

In [ ]:
# ── MANC (neuPrint) ──
manc_neuprint = Client(
    'neuprint.janelia.org',
    dataset='manc:v1.2.3',
    token=os.environ['NEUPRINT_APPLICATION_CREDENTIALS'],
)

In [ ]:
# ── Optic Lobe (neuPrint) ──
optic_lobe_neuprint = Client(
    'neuprint.janelia.org',
    dataset='optic-lobe:v1.1',
    token=os.environ['NEUPRINT_APPLICATION_CREDENTIALS'],
)

In [ ]:
# ── FIB-19 (neuPrint) ──
fib_19_neuprint = Client(
    'neuprint.janelia.org',
    dataset='fib19:v1.0',
    token=os.environ['NEUPRINT_APPLICATION_CREDENTIALS'],
)

In [ ]:
# ── Mushroom Body (neuPrint) ──
# NOTE: This node tracks the latest release and the exporter could not
# resolve which that is, so only the family is named. Set `dataset` to the
# exact release you mean before sharing the notebook.
mushroom_body_neuprint = Client(
    'neuprint.janelia.org',
    dataset='mushroombody',
    token=os.environ['NEUPRINT_APPLICATION_CREDENTIALS'],
)

In [ ]:
# ── Dataset (generic) ──
dataset_generic = Client(
    'neuprint.janelia.org',
    dataset='hemibrain:v1.1',
    token=os.environ['NEUPRINT_APPLICATION_CREDENTIALS'],
)

In [ ]:
# ── Upload Table ──
# NOTE: Coda stores an uploaded table in the browser, not in the graph, so
# the rows are not in this notebook. Point this at your copy of
# "annotations.csv".
upload_table = pd.read_csv('annotations.csv')
upload_table = upload_table.rename(columns={'root_id': 'neuronId'})
for _c in ['cluster']:
    upload_table[_c] = upload_table[_c].astype('string')

In [ ]:
# ── Table from URL ──
table_from_url = pd.read_csv('https://example.org/embedding.csv')
for _c in ['layer']:
    table_from_url[_c] = table_from_url[_c].astype('string')

## The transform chain

Everything below runs locally.

In [ ]:
# ── Find Neurons ──
find_neurons, _ = fetch_neurons(
    NeuronCriteria(
        type='^(?:LC.*)$',
        status=['Traced'],
        regex=True,
        client=hemibrain_neuprint,
    ),
    client=hemibrain_neuprint,
)
find_neurons = coda_neurons(find_neurons)
# NOTE: NeuronCriteria has no field for the rest of this node’s filters, so
# Coda applies them to the result instead. Same rows, one larger response.
find_neurons = find_neurons[(find_neurons['size'].notna() & (find_neurons['size'] >= 50000))]
find_neurons = find_neurons.head(200)

In [ ]:
# ── Input IDs ──
_ids = [1001, 1002, 1003]
input_ids, _ = fetch_neurons(NeuronCriteria(bodyId=_ids, client=hemibrain_neuprint), client=hemibrain_neuprint)
input_ids = coda_neurons(input_ids)

In [ ]:
# ── IDs from Label ──
_labels = ['DNp01', 'DNp02']
ids_from_label, _ = fetch_neurons(
    NeuronCriteria(type=_labels, status='Traced', client=hemibrain_neuprint),
    client=hemibrain_neuprint,
)
ids_from_label = coda_neurons(ids_from_label)

In [ ]:
# ── Explore Dataset ──
# NOTE: Explore Dataset downloads the whole neuron table once and searches
# it locally. On male-CNS that is around 165,000 rows; expect this cell to
# take a few seconds.
explore_dataset_all_, _ = fetch_neurons(NeuronCriteria(client=hemibrain_neuprint), client=hemibrain_neuprint)
explore_dataset_all_ = coda_neurons(explore_dataset_all_)

explore_dataset_hits = coda_search(explore_dataset_all_, 'status!=Traced pre>100')
# NOTE: Coda caps this at 500 hits and keeps the 500 most *relevant*; the
# relevance ranking is not ported, so this keeps the first 500 matches in
# table order instead. The rows may differ from the canvas.
explore_dataset_hits = explore_dataset_hits.head(500)

_selected_ids = [1001, 1005]
explore_dataset_selected = explore_dataset_all_[explore_dataset_all_['neuronId'].isin(_selected_ids)]

In [ ]:
# ── Cypher ──
cypher = fetch_custom(
    """
    MATCH (n:Neuron)
    WHERE n.pre > 1000
    RETURN n.bodyId, n.type
    """,
    client=hemibrain_neuprint,
)

In [ ]:
# ── ROI Completeness ──
roi_completeness = hemibrain_neuprint.fetch_roi_completeness().rename(
    columns={
        'roipre': 'pre',
        'roipost': 'post',
        'totalpre': 'totalPre',
        'totalpost': 'totalPost',
    }
)
roi_completeness['preCompleteness'] = (roi_completeness['pre'] / roi_completeness['totalPre']).where(roi_completeness['totalPre'] > 0)
roi_completeness['postCompleteness'] = (roi_completeness['post'] / roi_completeness['totalPost']).where(roi_completeness['totalPost'] > 0)
roi_completeness['primary'] = roi_completeness['roi'].isin(hemibrain_neuprint.primary_rois)

In [ ]:
# ── ROI Connectivity ──
roi_connectivity_links = hemibrain_neuprint.fetch_roi_connectivity().rename(
    columns={'from_roi': 'source', 'to_roi': 'target'}
)

_roi_connectivity_matrix_rois = sorted(set(roi_connectivity_links['source']) | set(roi_connectivity_links['target']))
roi_connectivity_matrix = (
    roi_connectivity_links
    .pivot_table(index='source', columns='target', values='weight', fill_value=0)
    .reindex(index=_roi_connectivity_matrix_rois, columns=_roi_connectivity_matrix_rois, fill_value=0)
)

In [ ]:
# ── Description ──
# NOTE: This card shows the dataset's published description and citation,
# which is prose rather than a step. Read it with `fetch_meta(client=...)`
# if you need it here.

In [ ]:
# ── Dataset Summary ──
dataset_summary_neurons, _ = fetch_neurons(NeuronCriteria(status='Traced', client=hemibrain_neuprint), client=hemibrain_neuprint)
dataset_summary_neurons = coda_neurons(dataset_summary_neurons)

# How many neurons carry each value of an attribute. dropna=False would count the
# missing ones as a category; the card reports them apart instead.
for _col in ['superclass', 'class', 'subclass', 'flow', 'somaSide', 'consensusNt', 'hemilineage', 'nerve']:
    if _col in dataset_summary_neurons.columns:
        print(dataset_summary_neurons[_col].value_counts().head(15))

dataset_summary_top_types = dataset_summary_neurons['type'].value_counts().head(15)

# Region completeness: traced synapses against the total present. The published list
# nests, so it is filtered to the primary set before anything is totalled.
dataset_summary_regions = hemibrain_neuprint.fetch_roi_completeness()
dataset_summary_regions = dataset_summary_regions[
    dataset_summary_regions['roi'].isin(hemibrain_neuprint.primary_rois)
].reset_index(drop=True)
dataset_summary_regions['preCompleteness'] = (
    dataset_summary_regions['roipre'] / dataset_summary_regions['totalpre']
).where(dataset_summary_regions['totalpre'] > 0)

In [ ]:
# ── ROI Viewer ──
# NOTE: Region meshes are OBJ bytes, one request each. neuPrint publishes
# them for visualization only — they are decimated display surfaces, so a
# volume measured off one is an approximation rather than a figure to quote.
# The published region list nests, so this walks the primary set that tiles the
# volume — the same default the card carries.
roi_viewer_meshes = {}
_skipped = []
for _roi in hemibrain_neuprint.primary_rois:
    try:
        roi_viewer_meshes[_roi] = hemibrain_neuprint.fetch_roi_mesh(_roi)
    except Exception:
        # No mesh published for this one. On male-CNS every such region is an
        # "-unspecified" bucket, which collects unassigned synapses and is not a shape.
        _skipped.append(_roi)

print(f"{len(roi_viewer_meshes)} region meshes"
      f" · {sum(len(_m) for _m in roi_viewer_meshes.values()) / 1e6:.1f} MB of OBJ"
      f" · {len(_skipped)} without one")

# Each value is the contents of an .obj file. To look at one:
#     open("ME(R).obj", "wb").write(roi_viewer_meshes["ME(R)"])
# or pass the bytes to trimesh / navis, neither of which this notebook imports.

In [ ]:
# ── ROI Meshes ──
# NOTE: Region meshes are one request each, and neuPrint publishes them for
# visualization only — decimated display surfaces, so a volume measured off
# one is an approximation rather than a figure to quote.
roi_meshes = []
_skipped = []
for _roi in ["EB", "FB"]:
    try:
        roi_meshes.append(neu.fetch_roi(_roi, client=hemibrain_neuprint))
    except Exception:
        # No mesh published for this one. On male-CNS every such region is an
        # "-unspecified" bucket, which collects unassigned synapses and is not a shape.
        _skipped.append(_roi)

print(f"{len(roi_meshes)} region meshes · {len(_skipped)} without one")

In [ ]:
# ── Landmark Transform ──
landmark_transform = navis.transforms.TPStransform(
    upload_table[['x', 'y', 'z']].values,
    upload_table[['x2', 'y2', 'z2']].values * 1000,
)

In [ ]:
# ── Google Sheet ──
google_sheet = coda_google_sheet(
    'https://docs.google.com/spreadsheets/d/1s0Pl9uTJ7Rl0Q1cQeXsp3s5kCsPRk9dU8jZ6yQnB4Vw/export?format=csv&gid=1874360847',
    id_column='root_id',
    columns=['cell_type', 'side'],
)
google_sheet = coda_join_annotations(upload_table, google_sheet)

In [ ]:
# ── Rename Columns ──
rename_columns = table_from_url.rename(
    columns={
        'root_id': 'neuronId',
        'cell_type': 'type',
    },
)

In [ ]:
# ── Connectivity ──
_neurons, _conn = fetch_adjacencies(
    NeuronCriteria(bodyId=find_neurons['neuronId'].tolist(), client=hemibrain_neuprint),
    None,
    min_total_weight=5,
    omit_rois=True,
    client=hemibrain_neuprint,
)
connectivity = (
    merge_neuron_properties(_neurons, _conn, ['type'])
    .rename(columns={
        'bodyId_pre': 'preId',
        'type_pre': 'preType',
        'bodyId_post': 'postId',
        'type_post': 'postType',
    })
    .assign(hop=1, direction='downstream')
)

In [ ]:
# ── Connectivity ──
connectivity_2 = coda_traverse_connectivity(
    find_neurons['neuronId'].tolist(),
    direction='both',
    hops=3,
    min_weight=10,
    client=hemibrain_neuprint,
)

In [ ]:
# ── ROI Counts ──
# NOTE: These counts nest: a synapse in LO(R) is counted again in its parent
# OL(R). Filter to `fetch_primary_rois(client=...)` before summing, or the
# totals roughly double.
_, roi_counts = fetch_neurons(
    NeuronCriteria(bodyId=find_neurons['neuronId'].tolist(), client=hemibrain_neuprint),
    client=hemibrain_neuprint,
)
roi_counts = coda_neurons(roi_counts)

In [ ]:
# ── Skeletons ──
skeletons = neu.fetch_skeletons(
    find_neurons['neuronId'].head(20).tolist(),
    heal=True,
    client=hemibrain_neuprint,
)

In [ ]:
# ── Meshes ──
# NOTE: Coda picks the level of detail from a triangle budget across the
# whole batch. navis takes a level, so this is a fixed one — raise `lod` for
# coarser, lower for finer.
meshes = neu.fetch_mesh_neuron(
    find_neurons['neuronId'].head(10).tolist(),
    lod=1,
    client=hemibrain_neuprint,
)

In [ ]:
# ── Synapses ──
synapses = fetch_synapses(
    NeuronCriteria(bodyId=find_neurons['neuronId'].tolist(), client=hemibrain_neuprint),
    SynapseCriteria(type='pre', primary_only=True, client=hemibrain_neuprint),
    client=hemibrain_neuprint,
)
synapses = coda_neurons(synapses)
synapses = coda_synapses(synapses)
# NOTE: neuprint-python calls the pre/post column "type"; Coda calls it
# "polarity" and keeps "type" for the cell type, which this frame does not
# carry — join it from a neuron table if you need it.

In [ ]:
# ── Scatter Plot ──
scatter_plot_out = find_neurons
scatter_plot_selected = scatter_plot_out[scatter_plot_out['neuronId'].isin([1001])]

plt.figure(figsize=(8, 6))
sns.scatterplot(data=scatter_plot_out, x='pre', y='post', alpha=0.8)
plt.xscale('log')
plt.yscale('log')
sns.regplot(data=scatter_plot_out, x='pre', y='post', scatter=False, ci=None)
plt.tight_layout()
plt.show()

In [ ]:
# ── Combine Columns ──
combine_columns = find_neurons.copy()
combine_columns['label'] = coda_combine(find_neurons, ['type', 'instance'])
combine_columns['label_from'] = coda_combine(find_neurons, ['type', 'instance'], source=True)

In [ ]:
# ── Group By ──
group_by = (
    find_neurons.groupby(['type'], dropna=False)
    .agg(n=('instance', 'size'), join_instance=('instance', coda_join))
    .reset_index()
)

In [ ]:
# ── Neuron Profile ──
neuron_profile_out = find_neurons
neuron_profile_current = neuron_profile_out[neuron_profile_out['neuronId'].isin([1001])]

_profile = coda_profile(
    [1001],
    client=hemibrain_neuprint,
    min_weight=1,
    top_n=10,
)

neuron_profile_summary = _profile['summary']
neuron_profile_upstream_types = _profile['upstream_types']
neuron_profile_downstream_types = _profile['downstream_types']
neuron_profile_top_upstream = _profile['top_upstream']
neuron_profile_top_downstream = _profile['top_downstream']
neuron_profile_regions = _profile['regions']
neuron_profile_hemispheres = _profile['hemispheres']

neuron_profile_summary

In [ ]:
# ── Neuroglancer ──
# TODO: This node builds a neuroglancer URL by editing the scene its dataset
# publishes. That scene is a fetch this translation does not make, so no
# link is built here.

In [ ]:
# ── Histogram ──
histogram_out = find_neurons
histogram_selected = histogram_out[
      ((histogram_out['pre'] >= 10) & (histogram_out['pre'] < 100))
    | ((histogram_out['pre'] >= 100) & (histogram_out['pre'] <= 1000))
]
# NOTE: Coda picks bins by Freedman–Diaconis capped at 80; seaborn’s
# `bins="auto"` takes the larger of Freedman–Diaconis and Sturges and has no
# cap, so the bar count can differ.

plt.figure(figsize=(8, 5))
sns.histplot(data=histogram_out, x='pre', hue='type', multiple='stack', bins='auto', log_scale=True)
plt.tight_layout()
plt.show()

In [ ]:
# ── Box Plot ──
box_plot_out = find_neurons
box_plot_selected = box_plot_out[box_plot_out['type'].astype(str).isin(['LC4'])]

plt.figure(figsize=(8, 6))
_order = box_plot_out['type'].value_counts().head(12).index
sns.violinplot(data=box_plot_out, x='pre', y='type', order=_order, inner='box')
plt.xscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# ── Box Plot ──
box_plot_2_out = find_neurons
# NOTE: No boxes are selected on the canvas, so Selected is empty.
box_plot_2_selected = box_plot_2_out.iloc[0:0]

plt.figure(figsize=(8, 6))
_order = box_plot_2_out['type'].value_counts().head(8).index
sns.boxplot(data=box_plot_2_out, y='pre', x='type', order=_order, whis=1.5, showfliers=False, boxprops=dict(alpha=0.35))
sns.swarmplot(data=box_plot_2_out, y='pre', x='type', order=_order, size=3)
# NOTE: Coda thins a swarm to 300 marks per group; seaborn plots every
# observation, so a large group is denser here than on the canvas.
plt.tight_layout()
plt.show()

In [ ]:
# ── Adjacency ──
_neurons, _conn = fetch_adjacencies(
    NeuronCriteria(bodyId=find_neurons['neuronId'].tolist(), client=hemibrain_neuprint),
    NeuronCriteria(bodyId=ids_from_label['neuronId'].tolist(), client=hemibrain_neuprint),
    client=hemibrain_neuprint,
)
_conn = merge_neuron_properties(_neurons, _conn, ['type'])
adjacency = connection_table_to_matrix(_conn, 'type', sort_by='type')

In [ ]:
# ── Paths ──
# NOTE: neuprint's `fetch_paths` returns every route within the hop budget.
# Coda additionally ranks them by their weakest link and keeps the strongest
# — that ranking is not reproduced here, so this is the unranked set.
paths_paths = fetch_paths(
    find_neurons['neuronId'].tolist(),
    ids_from_label['neuronId'].tolist(),
    min_weight=10,
    max_path_length=3,
    client=hemibrain_neuprint,
)

In [ ]:
# ── Filter ──
filter_ = connectivity[connectivity['weight'] >= 10]

In [ ]:
# ── Select One ──
select_one = skeletons[0:1]

In [ ]:
# ── Mirror Neurons ──
mirror_neurons = navis.mirror_brain(skeletons, template="JRCFIB2018F", warp=landmark_transform)

In [ ]:
# ── Transform Neurons ──
# NOTE: navis walks its full bridging graph here, where Coda fitted one
# spline through landmarks sampled from that same graph. The two agree to
# about 0.9 µm; this is the more accurate of the pair. Note the result is in
# micrometres, which navis carries on the neuron — anything downstream
# needing nanometres should convert rather than assume.
transform_neurons = navis.xform_brain(skeletons, source="JRCFIB2018F", target="JRC2018U")

In [ ]:
# ── Transform Neurons ──
transform_neurons_2 = navis.xform(skeletons, transform=landmark_transform)

In [ ]:
# ── Clean Skeletons ──
# NOTE: Coda holds coordinates in nanometres and these controls are
# micrometres, so the distances below are the card’s values times 1000. If
# your neurons are in other units — neuprintr and raw neuprint-python both
# return voxels — scale them to match.
_cleaned = []
for _neuron in navis.NeuronList(skeletons):
    nodes = _neuron.nodes.reset_index(drop=True)
    ids = np.arange(len(nodes))
    lookup = pd.Series(ids, index=nodes['node_id'])
    parents = np.where(
        nodes['parent_id'] >= 0, lookup.reindex(nodes['parent_id']).to_numpy(), -1
    ).astype(np.int64)
    coords = nodes[['x', 'y', 'z']].to_numpy(dtype=float)
    radii = nodes['radius'].to_numpy(dtype=float)
    parents = fastcore.heal_skeleton(
        ids, parents, coords, method='ALL',
        max_dist=10000,
    )
    coords = fastcore.smooth_skeleton_gaussian(ids, parents, coords, 2000)
    ids, parents, coords, source, alpha, _ = fastcore.resample_skeleton(
        ids, parents, coords, 1000
    )
    radii = radii[source[:, 0]] * (1 - alpha) + radii[source[:, 1]] * alpha
    _out = pd.DataFrame({
        'node_id': ids,
        'parent_id': parents,
        'x': coords[:, 0], 'y': coords[:, 1], 'z': coords[:, 2],
        'radius': radii,
    })
    _cleaned.append(navis.Skeleton(_out, id=_neuron.id, units=_neuron.units))
clean_skeletons = navis.NeuronList(_cleaned)

In [ ]:
# ── Clean Skeletons ──
# NOTE: Coda holds coordinates in nanometres and these controls are
# micrometres, so the distances below are the card’s values times 1000. If
# your neurons are in other units — neuprintr and raw neuprint-python both
# return voxels — scale them to match.
_cleaned = []
for _neuron in navis.NeuronList(skeletons):
    nodes = _neuron.nodes.reset_index(drop=True)
    ids = np.arange(len(nodes))
    lookup = pd.Series(ids, index=nodes['node_id'])
    parents = np.where(
        nodes['parent_id'] >= 0, lookup.reindex(nodes['parent_id']).to_numpy(), -1
    ).astype(np.int64)
    coords = nodes[['x', 'y', 'z']].to_numpy(dtype=float)
    radii = nodes['radius'].to_numpy(dtype=float)
    keep, parents, _, _ = fastcore.downsample_skeleton(ids, parents, 4)
    coords, radii, ids = coords[keep], radii[keep], keep
    _out = pd.DataFrame({
        'node_id': ids,
        'parent_id': parents,
        'x': coords[:, 0], 'y': coords[:, 1], 'z': coords[:, 2],
        'radius': radii,
    })
    _cleaned.append(navis.Skeleton(_out, id=_neuron.id, units=_neuron.units))
clean_skeletons_2 = navis.NeuronList(_cleaned)

In [ ]:
# ── NBLAST k-NN ──
# NOTE: NBLAST is calibrated in micrometres — navis: "Neurons should be in
# microns as NBLAST is optimized for that". This converts through the units
# navis carries on the neuron rather than assuming a factor.
nblast_k_nn_dp = navis.make_dotprops(
    skeletons.convert_units('um'),
    k=5,
    resample=1,
)
nblast_k_nn = navis.nblast_knn(
    nblast_k_nn_dp,
    k=5,
    scores='mean',
    n_candidates=200,
    format='long',
    normalized=True,
)
nblast_k_nn = nblast_k_nn.rename(columns={'query': 'queryId', 'target': 'targetId'})
# NOTE: Coda also carries "type" for each side as queryLabel / targetLabel.
# Join them back on from the neuron table if you need them here.

In [ ]:
# ── NBLAST ──
# NOTE: NBLAST is calibrated in micrometres — navis: "Neurons should be in
# microns as NBLAST is optimized for that". This converts through the units
# navis carries on the neuron rather than assuming a factor.
nblast_dp = navis.make_dotprops(
    skeletons.convert_units('um'),
    k=5,
    resample=1,
)
# NOTE: nblast_allbyall has no symmetry option, so the symmetric case goes
# through nblast(x, x, scores='mean'). Same scores, a little more work.
nblast = navis.nblast(
    nblast_dp,
    nblast_dp,
    scores='mean',
    normalized=True,
)
# NOTE: Coda labels the rows by "type"; this frame is indexed by neuron id,
# which is what every other navis call takes.

In [ ]:
# ── NBLAST ──
# NOTE: NBLAST is calibrated in micrometres — navis: "Neurons should be in
# microns as NBLAST is optimized for that". This converts through the units
# navis carries on the neuron rather than assuming a factor.
nblast_2_dp = navis.make_dotprops(
    skeletons.convert_units('um'),
    k=5,
)
nblast_2_target_dp = navis.make_dotprops(
    skeletons.convert_units('um'),
    k=5,
)
nblast_2 = navis.nblast(
    nblast_2_dp,
    nblast_2_target_dp,
    scores='forward',
    normalized=True,
    use_alpha=True,
)

In [ ]:
# ── 3D View ──
navis.plot3d([skeletons, *roi_meshes])

# NOTE: Nothing is picked in the viewer, so Selected is empty.
n_3d_view = pd.DataFrame({'neuronId': []})

In [ ]:
# ── Clean Meshes ──
# NOTE: Faces must be wound outward for Drop internal membrane: rays are
# fired into the hemisphere each normal points into, so an inward-wound mesh
# reads as entirely buried and comes back empty, and an inconsistently wound
# one loses healthy membrane without saying so.
_cleaned = []
for _neuron in navis.NeuronList(meshes):
    v, f = _neuron.vertices.astype(float), _neuron.faces.astype(np.uint32)
    v, f, _keep, _passes = fastcore.drop_internals(
        v, f,
        threshold=0.05,
        n_rays=16,
        iterations=3,
    )
    _he = fastcore.boundary_halfedges(f)
    if len(_he):
        _rings, _offsets = fastcore.trace_loops(_he)
        _caps = fastcore.triangulate_rings(_rings, _offsets, v)
        if len(_caps):
            f = np.vstack([f, _caps]).astype(np.uint32)
    v, f, _vmap = fastcore.simplify_mesh(f, v, ratio=0.25)
    v = fastcore.smooth_mesh(
        f, v,
        method='taubin',
        iterations=10,
        weights='cotangent',
        preserve_border=True,
        volume_correction=True,
    )
    _cleaned.append(navis.Mesh((v, f), id=_neuron.id, units=_neuron.units))
clean_meshes = navis.NeuronList(_cleaned)

In [ ]:
# ── Stack Neurons ──
stack_neurons = pd.concat(
    [
        synapses.assign(**{'batch': 'Run 1'}),
        synapses.assign(**{'batch': 'Run 2'}),
    ],
    ignore_index=True,
)

In [ ]:
# ── syNBLAST ──
# NOTE: syNBLAST is calibrated in micrometres like NBLAST, and
# neuprint-python returns synapse locations in raw voxels — 8 nm on the
# hemibrain. Check this factor against your dataset: nothing in the graph
# records it.
synblast_voxel_um = 8 / 1000

def synblast_connectors(frame):
    """One (N, 4) array of [x, y, z, type] per neuron, in first-appearance order."""
    out, labels = [], []
    for key, rows in frame.groupby('neuronId', sort=False):
        block = np.empty((len(rows), 4))
        block[:, :3] = rows[['x', 'y', 'z']].to_numpy() * synblast_voxel_um
        block[:, 3] = (rows['polarity'] == 'post').astype(float)
        out.append(block)
        labels.append(str(rows['type'].iloc[0]) if 'type' in rows.columns else str(key))
    return out, labels

synblast_groups, synblast_groups_labels = synblast_connectors(synapses)
# NOTE: Coda labels the rows by "type". neuprint-python's synapse frame
# carries fewer columns than Coda's does — notably not the cell type — so
# this falls back to the neuron id where the column is absent. Join it on
# from a neuron table to match.
synblast = fastcore.synblast(
    synblast_groups,
    None,
    by_type=True,
    normalize=True,
    symmetry='mean',
)
synblast = pd.DataFrame(
    synblast,
    index=synblast_groups_labels,
    columns=synblast_groups_labels,
)

In [ ]:
# ── Sort ──
sort = filter_.sort_values(by='weight', ascending=False, na_position='last', kind='stable')
sort = sort.head(500)

In [ ]:
# ── Stack Neurons ──
# NOTE: Coda adds the source as a column on the attribute table; navis
# carries it as an attribute on each neuron, which is what plot3d(color_by=)
# and NeuronList.summary() read.
for _n in skeletons:
    _n.side = 'Original'
for _n in mirror_neurons:
    _n.side = 'Mirrored'
stack_neurons_2 = navis.NeuronList([*skeletons, *mirror_neurons])

In [ ]:
# ── NBLAST Matches ──
nblast_matches_m = np.ascontiguousarray(nblast, dtype=float)
nblast_matches_groups = list(getattr(nblast, 'index', range(nblast_matches_m.shape[0])))
nblast_matches_targets = list(getattr(nblast, 'columns', range(nblast_matches_m.shape[1])))
_n = min(5, nblast_matches_m.shape[1] - 1)
_idx, _values = fastcore.top_matches(
    nblast_matches_m,
    n=_n,
    axis=0,
    distances=False,
    skip_self=True,
)
_rows, _n = _idx.shape
_group = np.repeat(np.arange(_rows), _n)
_rank = np.tile(np.arange(1, _n + 1), _rows)
_flat_idx, _flat_val = _idx.ravel(), _values.ravel()
_keep = _flat_idx >= 0
nblast_matches = pd.DataFrame({
    'query': np.asarray(nblast_matches_groups, dtype=object)[_group[_keep]],
    'target': np.asarray(nblast_matches_targets, dtype=object)[_flat_idx[_keep]],
    'rank': _rank[_keep],
    'score': _flat_val[_keep],
})

In [ ]:
# ── NBLAST Matches ──
nblast_matches_2_m = np.ascontiguousarray(nblast, dtype=float)
nblast_matches_2_groups = list(getattr(nblast, 'index', range(nblast_matches_2_m.shape[0])))
nblast_matches_2_targets = list(getattr(nblast, 'columns', range(nblast_matches_2_m.shape[1])))
# NOTE: Best means is on "from the matrix", which Coda answers by reading
# what the matrix says its cells are. A DataFrame has nowhere to carry that,
# so this assumes higher is better. Set distances=True below if this is a
# distance matrix.
_offsets, _flat_idx, _flat_val = fastcore.matches_above(
    nblast_matches_2_m,
    percentage=0.05,
    axis=0,
    distances=False,
    skip_self=True,
)
_counts = np.diff(_offsets)
_group = np.repeat(np.arange(len(_counts)), _counts)
_rank = np.concatenate([np.arange(1, c + 1) for c in _counts]) if len(_flat_idx) else _flat_idx
_keep = np.ones(len(_flat_idx), dtype=bool)
nblast_matches_2 = pd.DataFrame({
    'query': np.asarray(nblast_matches_2_groups, dtype=object)[_group[_keep]],
    'target': np.asarray(nblast_matches_2_targets, dtype=object)[_flat_idx[_keep]],
    'rank': _rank[_keep],
    'score': _flat_val[_keep],
})

In [ ]:
# ── NBLAST Matches ──
nblast_matches_3_m = np.ascontiguousarray(nblast, dtype=float)
nblast_matches_3_groups = list(getattr(nblast, 'index', range(nblast_matches_3_m.shape[0])))
nblast_matches_3_targets = list(getattr(nblast, 'columns', range(nblast_matches_3_m.shape[1])))
# NOTE: Best means is on "from the matrix", which Coda answers by reading
# what the matrix says its cells are. A DataFrame has nowhere to carry that,
# so this assumes higher is better. Set distances=True below if this is a
# distance matrix.
_counts = fastcore.count_matches(
    nblast_matches_3_m,
    threshold=0.4,
    axis=0,
    distances=False,
    skip_self=False,
)
nblast_matches_3 = pd.DataFrame({'query': nblast_matches_3_groups, 'matches': _counts})

In [ ]:
# ── Linkage ──
# NOTE: Coda runs navis-fastcore, whose linkage matrix is SciPy’s: checked
# against scipy.cluster.hierarchy.linkage on NBLAST-shaped matrices, merge
# order identical and heights agreeing to 1e-15. The fused pass fastcore
# uses saves memory, not accuracy.
_m = np.asarray(nblast, dtype=float)
_d = 1.0 - ((_m + _m.T) / 2)
linkage_tree = linkage(squareform(_d, checks=False), method='average')
linkage_tree_labels = list(getattr(nblast, 'index', range(len(_m))))
linkage_tree_order = leaves_list(linkage_tree)
linkage_tree_clusters = None
linkage_ordered = nblast.iloc[linkage_tree_order, linkage_tree_order]
# NOTE: Scores are similarities, so the distance is 1 − score. A matrix that
# carries distances already would be clustered as it stands.

In [ ]:
# ── Sample ──
sample = sort.iloc[coda_sample_rows(len(sort), 200, 7)]

In [ ]:
# ── Select One ──
select_one_2 = sort.iloc[3:4]

In [ ]:
# ── Cut Tree ──
_raw = np.asarray(cut_tree(linkage_tree, n_clusters=4).ravel())
# NOTE: Coda numbers clusters left to right as the dendrogram draws them, so
# the column reads against the picture. SciPy numbers them by its own
# bookkeeping — the grouping is identical either way; this renumbers so the
# two agree.
_renumber = {c: i + 1 for i, c in enumerate(dict.fromkeys(_raw[linkage_tree_order]))}
_cluster = [_renumber[c] for c in _raw]
_position = {int(obs): i for i, obs in enumerate(linkage_tree_order)}
cut_tree_clusters = pd.DataFrame({
    'label': linkage_tree_labels,
    'cluster': _cluster,
    'order': [_position[i] for i in range(len(linkage_tree_labels))],
})
cut_tree_clusters['size'] = cut_tree_clusters.groupby('cluster')['label'].transform('size')

cut_tree_tree = linkage_tree
cut_tree_tree_labels = linkage_tree_labels
cut_tree_tree_order = linkage_tree_order
cut_tree_tree_clusters = _cluster

In [ ]:
# ── Cut Tree ──
_raw = np.asarray(fcluster(linkage_tree, t=0.6, criterion='distance'))
# NOTE: Coda numbers clusters left to right as the dendrogram draws them, so
# the column reads against the picture. SciPy numbers them by its own
# bookkeeping — the grouping is identical either way; this renumbers so the
# two agree.
_renumber = {c: i + 1 for i, c in enumerate(dict.fromkeys(_raw[linkage_tree_order]))}
_cluster = [_renumber[c] for c in _raw]
_position = {int(obs): i for i, obs in enumerate(linkage_tree_order)}
cut_tree_2_clusters = pd.DataFrame({
    'label': linkage_tree_labels,
    'cluster': _cluster,
    'order': [_position[i] for i in range(len(linkage_tree_labels))],
})
cut_tree_2_clusters['size'] = cut_tree_2_clusters.groupby('cluster')['label'].transform('size')

cut_tree_2_tree = linkage_tree
cut_tree_2_tree_labels = linkage_tree_labels
cut_tree_2_tree_order = linkage_tree_order
cut_tree_2_tree_clusters = _cluster

In [ ]:
# ── Deduplicate ──
deduplicate = sample.drop_duplicates(subset=['preType', 'postType'], keep='last')

In [ ]:
# ── Dendrogram ──
dendrogram_out = cut_tree_tree
dendrogram_out_labels = cut_tree_tree_labels
dendrogram_out_order = cut_tree_tree_order
dendrogram_out_clusters = cut_tree_tree_clusters

plt.figure(figsize=(10, 6))
dendrogram(
    dendrogram_out,
    labels=dendrogram_out_labels,
    orientation='top',
)
plt.tight_layout()
plt.show()

_picked = [0, 2]
_position = {int(obs): i for i, obs in enumerate(dendrogram_out_order)}
_palette = ['#3987e5', '#d95926', '#199e70', '#c98500', '#d55181', '#008300', '#9085e9', '#e66767']
_cluster_of = lambda i: 0 if dendrogram_out_clusters is None else int(dendrogram_out_clusters[i])
_colour_of = lambda c: '#898781' if c <= 0 else _palette[(c - 1) % 8]
dendrogram_selected = pd.DataFrame({
    'label': [dendrogram_out_labels[i] for i in _picked],
    'order': [_position[i] for i in _picked],
    'cluster': [_cluster_of(i) for i in _picked],
    'color': [_colour_of(_cluster_of(i)) for i in _picked],
}).sort_values('order')

In [ ]:
# ── Clusters to Neurons ──
# NOTE: Coda matches labels as text, so both sides go through a string key:
# an NBLAST labelled by neuron id gives "722817260" against an int64 column,
# and merging those directly returns nothing at all.
_left = find_neurons.assign(_key=find_neurons['type'].astype(str))
_right = cut_tree_clusters.assign(_key=cut_tree_clusters['label'].astype(str))
_right = _right.drop_duplicates('_key').drop(columns=['label'])
clusters_to_neurons = _left.merge(_right, on='_key', how='inner', suffixes=('', '_c'))
clusters_to_neurons = clusters_to_neurons.drop(columns=['_key'])

In [ ]:
# ── Group By ──
group_by_2 = (
    deduplicate.groupby(['preType', 'postType'], dropna=False)
    .agg(n=('weight', 'size'), sum_weight=('weight', 'sum'))
    .reset_index()
)

In [ ]:
# ── Selected to Neurons ──
# NOTE: No neuron table is wired on the canvas, so the labels are read as
# neuron ids — which is what they are unless NBLAST was told to label by
# something else. Rows that are not usable ids are dropped, as they are in
# Coda.
selected_to_neurons = dendrogram_selected.copy()
selected_to_neurons['neuronId'] = pd.to_numeric(selected_to_neurons['label'], errors='coerce')
selected_to_neurons = selected_to_neurons[selected_to_neurons['neuronId'].notna()].drop(columns=['label'])
selected_to_neurons['neuronId'] = selected_to_neurons['neuronId'].astype('int64')
selected_to_neurons = selected_to_neurons[['neuronId'] + [c for c in selected_to_neurons.columns if c != 'neuronId']]

In [ ]:
# ── Select Columns ──
select_columns = group_by_2[['preType', 'postType', 'sum_weight']]

In [ ]:
# ── Bar Chart ──
bar_chart = group_by_2
bar_chart.plot.bar(x='preType', y='sum_weight', figsize=(10, 5), legend=False)
plt.ylabel('sum_weight')
plt.tight_layout()
plt.show()

In [ ]:
# ── Pie Chart ──
pie_chart_out = group_by_2
pie_chart_selected = pie_chart_out[pie_chart_out['preType'].astype(str).isin(['LC4'])]

_totals = pie_chart_out.groupby('preType')['sum_weight'].sum()
_totals = _totals.sort_values(ascending=False)
_top = _totals.head(6).copy()
if len(_totals) > 6:
    _top['Other'] = _totals.iloc[6:].sum()

plt.figure(figsize=(6, 6))
plt.pie(
    _top, labels=_top.index,
    autopct='%1.0f%%',
    wedgeprops=dict(width=0.42, edgecolor='white'),
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Table ──
table_out = group_by_2
table_filtered = table_out[
      (table_out['sum_weight'].notna() & (table_out['sum_weight'] >= 10))
    & table_out['preType'].astype(str).str.contains('LC', regex=True, case=False, na=False)
    & table_out['postType'].astype(str).str.contains('^DN', regex=True, case=False, na=False)
    & ~(table_out['n'].notna() & (table_out['n'] == 1))
]
# NOTE: Coda matches these regexes with JavaScript semantics and pandas uses
# Python `re`. The two agree on ordinary patterns and differ on lookbehind
# and named groups.
table_filtered

In [ ]:
# ── Build Network ──
_links = (
    group_by_2.groupby(['preType', 'postType'], dropna=False)
    .agg(weight=('sum_weight', 'sum'), edges=('sum_weight', 'size'))
    .reset_index()
)
_links = _links[_links['weight'] >= 5]
build_network = nx.from_pandas_edgelist(
    _links,
    source='preType',
    target='postType',
    edge_attr=['weight', 'edges'],
    create_using=nx.DiGraph if True else nx.Graph,
)

In [ ]:
# ── Join ──
# NOTE: Coda keeps the first matching row from the other table, so a
# duplicated key annotates rather than multiplies. `drop_duplicates` is what
# reproduces that — without it `merge` returns the cross product of every
# matching pair.
join = pd.merge(
    select_columns,
    group_by_2.drop_duplicates(subset=['preType']),
    how='left',
    left_on='preType',
    right_on='preType',
    suffixes=('', '_r'),
)

In [ ]:
# ── Join ──
# NOTE: Coda keeps the first matching row from the other table, so a
# duplicated key annotates rather than multiplies. `drop_duplicates` is what
# reproduces that — without it `merge` returns the cross product of every
# matching pair.
# NOTE: Coda publishes **one** key column, filled from whichever side the
# row came from — `dplyr::full_join`'s shape rather than pandas'. Where the
# two keys are named differently pandas keeps both, so the right one is
# renamed out of the way and dropped, after filling 'preType' from it.
join_2 = pd.merge(
    select_columns,
    group_by_2.drop_duplicates(subset=['postType']).rename(columns={'postType': '_coda_join_key'}),
    how='outer',
    left_on='preType',
    right_on='_coda_join_key',
    suffixes=('', '_r'),
)
join_2['preType'] = join_2['preType'].fillna(join_2['_coda_join_key'])
join_2 = join_2.drop(columns=['_coda_join_key'])

In [ ]:
# ── Join ──
# NOTE: Coda keeps the first matching row from the other table, so a
# duplicated key annotates rather than multiplies. `drop_duplicates` is what
# reproduces that — without it `merge` returns the cross product of every
# matching pair.
# NOTE: Coda publishes **one** key column, filled from whichever side the
# row came from — `dplyr::full_join`'s shape rather than pandas'. Where the
# two keys are named differently pandas keeps both, so the right one is
# renamed out of the way and dropped, after filling 'preType' from it.
join_3 = pd.merge(
    select_columns.drop_duplicates(subset=['preType']),
    group_by_2.rename(columns={'postType': '_coda_join_key'}),
    how='right',
    left_on='preType',
    right_on='_coda_join_key',
    suffixes=('', '_r'),
)
join_3['preType'] = join_3['preType'].fillna(join_3['_coda_join_key'])
join_3 = join_3.drop(columns=['_coda_join_key'])

In [ ]:
# ── Network Viewer ──
network_viewer_out = build_network.copy()
network_viewer_out.remove_edges_from([
    (u, v) for u, v, w in network_viewer_out.edges(data='weight')
    if (w or 0) < 10
])
network_viewer_out.remove_nodes_from(list(nx.isolates(network_viewer_out)))

# NOTE: Coda draws this with ForceAtlas2 in the browser. networkx has no
# equivalent, so the graph is handed over and the layout is yours to pick —
# uncomment one.
# pos = nx.spring_layout(network_viewer_out, weight='weight')
# pos = nx.kamada_kawai_layout(network_viewer_out, weight='weight')
# pos = nx.nx_agraph.graphviz_layout(network_viewer_out, prog='dot')  # needs pygraphviz
#
# nx.draw_networkx(
#     network_viewer_out, pos,
#     node_size=40,
#     width=[d['weight'] / 20 for _, _, d in network_viewer_out.edges(data=True)],
#     with_labels=True, font_size=7,
# )

In [ ]:
# ── Stack Tables ──
stack_tables = pd.concat(
    [
        join.assign(**{'origin': 'Direct'}),
        connectivity_2.assign(**{'origin': 'Indirect'}),
    ],
    ignore_index=True,
)

In [ ]:
# ── Pivot ──
pivot_matrix = stack_tables.pivot_table(
    index='preType',
    columns='postType',
    values='weight',
    aggfunc='sum',
    fill_value=0,
)
pivot_table = pivot_matrix.reset_index()

In [ ]:
# ── Muted step ──
# Muted on the canvas, so it produced nothing and nothing downstream of it
# ran. Left here rather than dropped, because a node missing from the
# notebook and a node deliberately switched off look identical otherwise.

In [ ]:
# ── Normalize ──
normalize = pivot_matrix.div(pivot_matrix.sum(axis=1), axis=0).fillna(0)

In [ ]:
# ── Table ──
table_2_out = pivot_table
table_2_filtered = table_2_out
table_2_out

In [ ]:
# ── Heatmap ──
heatmap = normalize
_plot = heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(_plot, cmap='rocket')
plt.tight_layout()
plt.show()

In [ ]:
# ── Download ──
download = table_2_out
download.to_csv('partners.csv', index=False)